# Capa Oro - FAIAD (SparkSQL)

Full process (CTAS): lee Bronze (esquema `dbo`, tablas del shortcut) y materializa Gold (esquema `gold`) en tablas Delta para el modelo Direct Lake.

REQUISITO: adjuntar el lakehouse `lh_FAIAD` a este notebook (panel izquierdo > Add lakehouse).

Nota: si los nombres `dbo.Tabla` / `gold.Tabla` no resuelven en tu ambiente, ajustar el calificador segun como exponga el catalogo el lakehouse adjunto.

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.Geo AS
SELECT
    c.CityID,
    c.CityName,
    c.LatestRecordedPopulation,
    s.StateProvinceCode,
    s.StateProvinceName,
    s.SalesTerritory,
    co.CountryName,
    co.FormalName,
    co.IsoAlpha3Code,
    co.IsoNumericCode,
    co.CountryType,
    co.Continent,
    co.Region,
    co.Subregion
FROM dbo.Cities AS c
INNER JOIN dbo.States AS s ON c.StateProvinceID = s.StateProvinceID
INNER JOIN dbo.Countries AS co ON s.CountryID = co.CountryID;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.Product AS
SELECT
    pi.StockItemID,
    pi.StockItemName,
    pi.SupplierID,
    pi.Size,
    pi.IsChillerStock,
    pi.TaxRate,
    pi.UnitPrice,
    pi.RecommendedRetailPrice,
    pi.TypicalWeightPerUnit,
    pg.StockGroupName
FROM dbo.ProductItem AS pi
LEFT JOIN dbo.ProductItemGroup AS pig ON pi.StockItemID = pig.StockItemID
LEFT JOIN dbo.ProductGroups AS pg ON pig.StockGroupID = pg.StockGroupID;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.Reseller AS
SELECT
    cu.ResellerID,
    cu.ResellerName,
    cu.PostalCityID,
    cu.PhoneNumber,
    cu.FaxNumber,
    cu.WebsiteURL,
    cu.DeliveryAddressLine1,
    cu.DeliveryAddressLine2,
    cu.DeliveryPostalCode,
    cu.PostalAddressLine1,
    cu.PostalAddressLine2,
    cu.PostalPostalCode,
    bg.BuyingGroupName AS ResellerCompany
FROM dbo.Customers AS cu
INNER JOIN dbo.BuyingGroups AS bg ON cu.BuyingGroupID = bg.BuyingGroupID;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.Sales AS
SELECT
    il.InvoiceLineID,
    il.InvoiceID,
    il.StockItemID,
    il.Quantity,
    il.UnitPrice,
    il.TaxRate,
    il.TaxAmount,
    il.LineProfit,
    il.ExtendedPrice,
    inv.CustomerID AS ResellerID,
    inv.SalespersonPersonID,
    CAST(inv.InvoiceDate AS DATE) AS InvoiceDate,
    il.ExtendedPrice - il.TaxAmount AS SalesAmount
FROM dbo.InvoiceLineItems AS il
INNER JOIN dbo.Invoices AS inv ON il.InvoiceID = inv.InvoiceID
WHERE inv.CustomerID IN (SELECT ResellerID FROM gold.Reseller);

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold.Dates AS
SELECT * FROM dbo.`Date`;

## Pendientes (completar tras capturar esquemas del Lab 4)

Las siguientes tablas dependen de los dataflows / fuentes del Lab 4. Capturar el esquema real en el dry-run y completar con el mismo patron `CREATE OR REPLACE TABLE gold.<Tabla> AS SELECT ... FROM dbo.<origen>`.

- gold.Customer
- gold.People
- gold.PO
- gold.Supplier
- gold.Invoices  (tabla del WOW; agregar los invoices de mayo aqui y re-ejecutar el notebook)

In [ ]:
%%sql
-- PLACEHOLDER - completar con el esquema real del Lab 4
-- CREATE OR REPLACE TABLE gold.Customer AS SELECT * FROM dbo.Customer;
-- CREATE OR REPLACE TABLE gold.People   AS SELECT * FROM dbo.People;
-- CREATE OR REPLACE TABLE gold.PO       AS SELECT * FROM dbo.PurchaseOrders;
-- CREATE OR REPLACE TABLE gold.Supplier AS SELECT * FROM dbo.Supplier;
-- CREATE OR REPLACE TABLE gold.Invoices AS SELECT * FROM dbo.Invoices;
SELECT 'completar celdas pendientes' AS nota;